In [1]:
import pandas as pd
import numpy as np
import requests
import io
from collections import Counter

In [2]:
# original data used for xmatch
data_from_drive = pd.read_csv('../data/redo_data_Sep2026/full_galactic_table.csv')

In [3]:
data_from_drive.shape

(12147664, 3)

In [4]:
# results from the xmatch,unique objects
data = pd.read_parquet('../data/redo_data_Sep2026/fxmatch_ztf_2026-10-07_943076/')

In [5]:
data.shape

(1723098, 11)

In [6]:
data_obj = data.groupby('objectId').agg(list)

# count the number of points in the filter with max number of points
data_obj['npoints'] = data_obj['fid'].apply(lambda x: max(Counter(x).values(), default=0))

In [7]:
# remove all objects with less than  points
data_obj_long = data_obj[data_obj['npoints'] > 3].copy()

In [8]:
data_obj_long.shape

(29428, 11)

In [9]:
#percentage of xmatch returned
np.round(100*data_obj_long.shape[0]/data_from_drive.shape[0], 3)

np.float64(0.242)

In [10]:
data_obj_long['final_class'] = data_obj_long["finkclass"].apply(lambda x: Counter(x).most_common(1)[0][0] if len(x) > 0 else None)

In [11]:
groups, pop = np.unique(data_obj_long['final_class'].values, return_counts=True)
indx = np.argsort(pop)
for i in range(indx.shape[0]):
    print(groups[indx[i]], pop[indx[i]])

Ae*_Candidate 1
BLLac 1
BSS_Candidate 1
CataclyV*_Candidate 1
Candidate_WD* 1
Candidate_XB* 1
Candidate_CV* 1
Candidate_C* 1
IrregularV* 1
GinGroup 1
DkNeb 1
ClassicalCep 1
QSO_Candidate 1
PulsV*RVTau 1
Planet?_Candidate 1
Planet 1
RfNeb 1
Symbiotic* 1
Radio(cm) 1
RGB*_Candidate 1
alf2CVnV* 1
XB 1
AGN_Candidate 2
Candidate_AGB* 2
Cl* 2
Low-Mass* 2
EclBin_Candidate 2
Hsd_Candidate 2
ChemPec* 2
Candidate_RGB* 2
Cepheid_Candidate 2
AGN 2
Blue 2
multiple_object 2
V*? 2
WR* 2
SN 2
RotV*alf2CVn 2
PartofG 2
PN?_Candidate 2
AGB*_Candidate 2
S* 3
LongPeriodV*_Candidate 3
IR 3
TTau*_Candidate 3
Pec* 3
RedSG* 4
post-AGB* 4
MIR 4
GlCl 4
HB*_Candidate 4
RRLyr_Candidate 5
Irregular_V* 5
PulsV*delSct 6
LongPeriodV* 7
OrionV* 8
Variable* 8
gammaDor 8
Candidate_HB* 8
NIR 9
BlueStraggler 9
EmLine* 10
CataclyV* 11
Candidate_TTau* 11
Ae* 11
pulsV*SX 13
X 13
Seyfert_1 14
EllipVar 14
TTauri* 15
PM* 16
** 18
EclBin 18
SB* 19
C* 19
SN candidate 21
Cepheid 26
AGB* 26
PulsV*WVir 27
EB*_Candidate 27
Eruptive* 30

In [12]:
data_obj_long['id_final'] = data_obj_long["full_id"].apply(lambda x: Counter(x).most_common(1)[0][0] if len(x) > 0 else None)

In [13]:
# add column with folder name
name_list = []

for i in range(data_obj_long.shape[0]):
    name = data_obj_long.iloc[i]['id_final'].split('/', 1)[0]

    if 'LEGUSv3' in name:
        name_use = 'LEGUSv3'
    elif 'nan' in name:
        name_use = 'nan'
    else:
        name_use = name

    name_list.append(name_use)

data_obj_long['folder'] = name_list

In [15]:
# calculate number of xmatches per folder
folder, pop =  np.unique(data_obj_long['folder'].values, return_counts=True)

print('Folder \t n_matches')
for i in range(len(folder)):
    print(folder[i], '\t', pop[i])

Folder 	 n_matches
GalGCs_GAIAFlags 	 9
LEGUSv3 	 4
PHANGS 	 2
Sept13 	 1
galactic 	 2563
nan 	 3
openclusters 	 26840
ymcs 	 6


In [18]:
data_obj_long[data_obj_long['id_final'].apply(lambda x: 'nan' in x)].index

Index(['ZTF18aakpjyg', 'ZTF18abauhfb', 'ZTF19aaufzcu'], dtype='object', name='objectId')

In [19]:
# nan are a result of a problem in the database, these were checked by hand -- nothing happening
for name in data_obj_long[data_obj_long['id_final'].apply(lambda x: 'nan' in x)].index:
    print('https://ztf.fink-portal.org/' + name)

https://ztf.fink-portal.org/ZTF18aakpjyg
https://ztf.fink-portal.org/ZTF18abauhfb
https://ztf.fink-portal.org/ZTF19aaufzcu


In [38]:
subset

Index(['ZTF18aaklptu'], dtype='object', name='objectId')

In [40]:
# check by hand Sept13 folder
mask_sep13 = data_obj_long['id_final'].apply(lambda x: 'Sept13' in x)
subset = data_obj_long[mask_sep13]
for i in range(subset.shape[0]):
    print('https://ztf.fink-portal.org/' + subset.index[i], subset.iloc[i]['final_class'])

https://ztf.fink-portal.org/ZTF18aaklptu GinGroup


In [47]:
# check by hand PHANGS folder
mask_phangs = data_obj_long['id_final'].apply(lambda x: 'PHANGS' in x)
subset = data_obj_long[mask_phangs]
for i in range(subset.shape[0]):
    print('https://ztf.fink-portal.org/' + subset.index[i], subset.iloc[i]['final_class'])

https://ztf.fink-portal.org/ZTF20aaelulu SN
https://ztf.fink-portal.org/ZTF23aaeewkx Unknown


In [50]:
# checked by hand GalGCs_GAIAFlags folder
subset = data_obj_long[data_obj_long['id_final'].apply(lambda x: 'GalGCs_GAIAFlags' in x)]
for i in range(subset.shape[0]):
    print('https://ztf.fink-portal.org/' + subset.index[i], subset.iloc[i]['final_class'])

https://ztf.fink-portal.org/ZTF18aaajfuh RRLyr
https://ztf.fink-portal.org/ZTF18aaakbxb RRLyr
https://ztf.fink-portal.org/ZTF18aabcxog RRLyr
https://ztf.fink-portal.org/ZTF18aablcbe RRLyr
https://ztf.fink-portal.org/ZTF18acbytzf Unknown
https://ztf.fink-portal.org/ZTF18accncek QSO
https://ztf.fink-portal.org/ZTF18aceohig RRLyr
https://ztf.fink-portal.org/ZTF19abatuvw RRLyr
https://ztf.fink-portal.org/ZTF21acjlrqr Unknown


In [52]:
# checked by hand LEGUSv3 folders
subset = data_obj_long[data_obj_long['id_final'].apply(lambda x: 'LEGUSv3' in x)]
for i in range(subset.shape[0]):
    print('https://ztf.fink-portal.org/' + subset.index[i], subset.iloc[i]['final_class'])

https://ztf.fink-portal.org/ZTF18aadkuel Unknown
https://ztf.fink-portal.org/ZTF20aaqhshy Cl*
https://ztf.fink-portal.org/ZTF23aaazair Unknown
https://ztf.fink-portal.org/ZTF23aaklqou SN


In [55]:
# checked by hand ymcs folders
subset = data_obj_long[data_obj_long['id_final'].apply(lambda x: 'ymcs' in x)]
for i in range(subset.shape[0]):
    print('https://ztf.fink-portal.org/' + subset.index[i], subset.iloc[i]['final_class'])

https://ztf.fink-portal.org/ZTF17aaajnki PartofG
https://ztf.fink-portal.org/ZTF19acdiuzc Unknown
https://ztf.fink-portal.org/ZTF19acynape Unknown
https://ztf.fink-portal.org/ZTF19addoreu PartofG
https://ztf.fink-portal.org/ZTF21acmhmaa Unknown
https://ztf.fink-portal.org/ZTF22abpvjcd Unknown


In [53]:
# focus on the galactic folder
data_galactic = data_obj_long[data_obj_long['folder'].apply(lambda x: 'galactic' in x)]

In [54]:
class_gal, popo_gal = np.unique(data_galactic['final_class'].values, return_counts=True)

for i in range(class_gal.shape[0]):
    print(class_gal[i], '\t', popo_gal[i])

** 	 1
AGB* 	 14
BlueStraggler 	 2
Candidate_EB* 	 3
Candidate_HB* 	 2
Candidate_LP* 	 19
Candidate_RGB* 	 1
Candidate_RRLyr 	 36
Candidate_XB* 	 1
Candidate_YSO 	 1
CataclyV* 	 4
Cepheid 	 12
ChemPec* 	 1
EB* 	 82
EB*_Candidate 	 2
EllipVar 	 2
GlCl 	 4
HB* 	 43
HB*_Candidate 	 1
Hsd_Candidate 	 1
Irregular_V* 	 1
LP*_Candidate 	 71
LPV* 	 104
Mira 	 32
NIR 	 4
PulsV* 	 5
PulsV*WVir 	 15
QSO 	 17
RGB* 	 99
RRLyr 	 722
RRLyr_Candidate 	 5
RotV* 	 2
SN candidate 	 4
Seyfert_1 	 3
Star 	 213
Symbiotic* 	 1
TTau* 	 1
Unknown 	 937
V* 	 78
V*? 	 2
X 	 3
XB 	 1
deltaCep 	 1
low-mass* 	 1
post-AGB* 	 1
pulsV*SX 	 8


In [60]:
data_galactic.to_parquet('../data/redo_data_Sep2026/long_obj_galactic.parquet')

In [56]:
# focus on the opencluster folder
data_openclusters = data_obj_long[data_obj_long['folder'].apply(lambda x: 'openclusters' in x)]

In [57]:
class_oc, pop_oc = np.unique(data_openclusters['final_class'].values, return_counts=True)

for i in range(class_oc.shape[0]):
    print(class_oc[i], '\t', pop_oc[i])

** 	 17
AGB* 	 12
AGB*_Candidate 	 2
AGN 	 2
AGN_Candidate 	 2
Ae* 	 11
Ae*_Candidate 	 1
BLLac 	 1
BSS_Candidate 	 1
BYDra 	 123
BYDraV* 	 104
Be* 	 78
Blue 	 2
BlueStraggler 	 7
C* 	 19
Candidate_AGB* 	 2
Candidate_C* 	 1
Candidate_CV* 	 1
Candidate_EB* 	 89
Candidate_HB* 	 6
Candidate_LP* 	 34
Candidate_RGB* 	 1
Candidate_RRLyr 	 23
Candidate_TTau* 	 11
Candidate_WD* 	 1
Candidate_YSO 	 917
CataclyV* 	 7
CataclyV*_Candidate 	 1
Cepheid 	 14
Cepheid_Candidate 	 2
ChemPec* 	 1
ClassicalCep 	 1
DkNeb 	 1
EB* 	 266
EB*_Candidate 	 25
EclBin 	 18
EclBin_Candidate 	 2
EllipVar 	 12
Em* 	 402
EmLine* 	 10
Eruptive* 	 30
HB* 	 27
HB*_Candidate 	 3
Hsd_Candidate 	 1
IR 	 3
IrregularV* 	 1
Irregular_V* 	 4
LP*_Candidate 	 31
LPV* 	 66
LongPeriodV* 	 7
LongPeriodV*_Candidate 	 3
Low-Mass* 	 2
MIR 	 4
Mira 	 4
NIR 	 5
OrionV* 	 8
Orion_V* 	 265
PM* 	 16
PN?_Candidate 	 2
Pec* 	 3
Planet 	 1
Planet?_Candidate 	 1
PulsV* 	 49
PulsV*RVTau 	 1
PulsV*WVir 	 12
PulsV*delSct 	 6
QSO 	 73
QSO_Candidate

In [59]:
data_openclusters.to_parquet('../data/redo_data_Sep2026/long_obj_openclusters.parquet')